# ML Dataset

Phase 4 — aggregate `fact_trips` to one row per station per hour and write
`data/featured/`. See [01-pyspark-warehouse.md](../../docs/de/01-pyspark-warehouse.md).


In [ ]:
# session + paths
import math

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("ml-dataset")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    # the worker has 2 cores; without a cap one idle session starves the rest
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    # zero-fill produces ~10M rows from 5.3M trips; give the shuffle room
    .config("spark.sql.shuffle.partitions", 64)
    .getOrCreate()
)

print("app id :", spark.sparkContext.applicationId)

WAREHOUSE = "/opt/data/warehouse"
FEATURED = "/opt/data/featured"

fact_trips = spark.read.parquet(f"{WAREHOUSE}/fact_trips")
dim_station = spark.read.parquet(f"{WAREHOUSE}/dim_station")

print(f"fact_trips  : {fact_trips.count():,} rows")
print(f"dim_station : {dim_station.count():,} stations")


## Base aggregation and zero-fill

`trip_count` per station per hour, cross-joined against the full hourly calendar
so hours with no departures are `0` rather than missing rows.


In [ ]:
# base aggregation — departures per station per hour
departures = fact_trips.groupBy("start_station_id", "start_date", "start_hour").agg(
    F.count("*").alias("trip_count"),
    # member_ratio needs the member share of the same hour's trips
    F.sum(F.when(F.col("user_type_id") == 1, 1).otherwise(0)).alias("member_trips"),
)

print(f"observed station-hours : {departures.count():,}")

# the hourly calendar spanning the data, as a dense sequence of dates
bounds = fact_trips.agg(
    F.min("start_date").alias("min_date"), F.max("start_date").alias("max_date")
).first()
print(f"date range : {bounds.min_date} .. {bounds.max_date}")

calendar = (
    spark.sql(
        f"SELECT explode(sequence("
        f"  to_date('{bounds.min_date}'), to_date('{bounds.max_date}'), interval 1 day"
        f")) AS start_date"
    )
    .crossJoin(spark.range(24).withColumnRenamed("id", "start_hour"))
    .withColumn("start_hour", F.col("start_hour").cast("int"))
)

# every station x every hour — the grid the model predicts over
grid = dim_station.select("station_id").crossJoin(calendar)

hourly = (
    grid.join(
        departures,
        (grid.station_id == departures.start_station_id)
        & (grid.start_date == departures.start_date)
        & (grid.start_hour == departures.start_hour),
        "left",
    )
    .select(
        grid.station_id,
        grid.start_date,
        grid.start_hour,
        F.coalesce(departures.trip_count, F.lit(0)).alias("trip_count"),
        F.coalesce(departures.member_trips, F.lit(0)).alias("member_trips"),
    )
    .cache()
)

total = hourly.count()
zeros = hourly.where(F.col("trip_count") == 0).count()
print(f"grid rows  : {total:,}  ({zeros:,} zero-filled, {zeros / total:.1%})")


## Calendar, holiday, and cyclical features


In [ ]:
# calendar + holiday + cyclical encodings
# Ontario statutory holidays, derived rather than hard-coded: Family Day is the
# 3rd Monday of February, Victoria Day the Monday before May 25, and so on, so
# the dates move year to year. An earlier hard-coded list covered only 2019-2020,
# which left is_holiday silently 0 for 2021-2023 — train and test then disagreed
# on what the flag meant, so the model could not use it at all.
from holidays import country_holidays

YEARS_COVERED = list(range(bounds.min_date.year, bounds.max_date.year + 1))
ONTARIO_HOLIDAYS = sorted(
    d.isoformat() for d in country_holidays("CA", subdiv="ON", years=YEARS_COVERED)
)
print(f"holiday years : {YEARS_COVERED}")
print(f"holiday dates : {len(ONTARIO_HOLIDAYS)} across {len(YEARS_COVERED)} years")
assert len(ONTARIO_HOLIDAYS) >= 9 * len(YEARS_COVERED), "holiday coverage looks short"

featured = (
    hourly.withColumn("hour", F.col("start_hour"))
    .withColumn("year", F.year("start_date"))
    .withColumn("month", F.month("start_date"))
    .withColumn("quarter", F.quarter("start_date"))
    # dayofweek() is 1=Sunday; the warehouse convention is 1=Monday .. 7=Sunday
    .withColumn("day_of_week", ((F.dayofweek("start_date") + 5) % 7 + 1).cast("int"))
)

featured = (
    featured.withColumn(
        "is_weekend", F.col("day_of_week").isin(6, 7).cast("int")
    )
    .withColumn(
        "is_holiday",
        F.col("start_date").isin([F.to_date(F.lit(d)) for d in ONTARIO_HOLIDAYS]).cast("int"),
    )
    # cyclical encodings so 23:00 and 00:00 are adjacent, likewise Sun and Mon
    .withColumn("hour_sin", F.sin(2 * math.pi * F.col("hour") / 24))
    .withColumn("hour_cos", F.cos(2 * math.pi * F.col("hour") / 24))
    .withColumn("dow_sin", F.sin(2 * math.pi * (F.col("day_of_week") - 1) / 7))
    .withColumn("dow_cos", F.cos(2 * math.pi * (F.col("day_of_week") - 1) / 7))
)

# every year in the data must carry holidays, or the flag is untrainable
per_year = (
    featured.where(F.col("is_holiday") == 1)
    .select("year", "start_date")
    .distinct()
    .groupBy("year")
    .count()
    .orderBy("year")
)
per_year.show()
assert per_year.count() == len(YEARS_COVERED), "some years have no holidays flagged"

featured.select(
    "start_date", "hour", "day_of_week", "is_weekend", "is_holiday"
).where(F.col("is_holiday") == 1).show(5)

## Lags, rolling means, and member ratio

Windows are partitioned by station and ordered by timestamp, so a row only ever
sees earlier hours — no future information leaks in.


In [ ]:
# lags, rolling means, member ratio
# an integer hour index makes the window ordering explicit and gap-free, so
# rangeBetween counts real elapsed hours rather than row positions
featured = featured.withColumn(
    "ts", F.to_timestamp(F.col("start_date")) + F.expr("make_interval(0,0,0,0,hour,0,0)")
).withColumn("hour_index", (F.col("ts").cast("long") / 3600).cast("long"))

by_station = Window.partitionBy("station_id").orderBy("hour_index")


def trailing(hours: int) -> Window:
    """Window over the previous `hours` hours, excluding the current row."""
    return by_station.rangeBetween(-hours, -1)


featured = (
    featured.withColumn("lag_1h", F.lag("trip_count", 1).over(by_station))
    .withColumn("lag_24h", F.lag("trip_count", 24).over(by_station))
    .withColumn("lag_168h", F.lag("trip_count", 168).over(by_station))
    .withColumn("roll_mean_24h", F.avg("trip_count").over(trailing(24)))
    .withColumn("roll_mean_168h", F.avg("trip_count").over(trailing(168)))
    # member share of the hour's trips; 0 trips means no ratio to speak of
    .withColumn(
        "member_ratio",
        F.when(F.col("trip_count") > 0, F.col("member_trips") / F.col("trip_count")),
    )
)

ML_COLUMNS = [
    "station_id",
    "start_date",
    "hour",
    "trip_count",  # target
    "year",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
    "member_ratio",
]

ml_dataset = featured.select(*ML_COLUMNS).cache()

print(f"rows    : {ml_dataset.count():,}")
print(f"columns : {len(ML_COLUMNS)}")
ml_dataset.orderBy("station_id", "start_date", "hour").show(5)


## Write and validate


In [ ]:
# write + validate
# Full replace: partitionOverwriteMode=dynamic only rewrites the partitions this
# write touches, so an earlier run's part-files survive and the next read unions
# both generations. That is the defect that inflated dim_station and, through the
# cross-join, this table.
hadoop = spark._jvm.org.apache.hadoop.fs
hadoop.FileSystem.get(spark._jsc.hadoopConfiguration()).delete(
    hadoop.Path(f"{FEATURED}/hourly"), True
)
(
    ml_dataset.write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(f"{FEATURED}/hourly")
)

written = spark.read.parquet(f"{FEATURED}/hourly")
n = written.count()
print(f"written : {n:,} rows -> {FEATURED}/hourly\n")

checks = []

# The grid must be exactly (distinct stations) x (distinct station-hours).
# dim_station.count() is NOT usable here: if the dimension ever carries a
# duplicated station_id, both sides of the comparison inflate together and the
# check passes on corrupt data. Count distinct ids explicitly.
stations = dim_station.select("station_id").distinct().count()
expected = written.select("start_date", "hour").distinct().count() * stations
print(f"distinct stations : {stations:,}")
print(f"expected rows     : {expected:,}")
checks.append(("grid is complete", n == expected))

# One row per station-hour. This is the check that actually catches a fan-out,
# so assert it rather than only reporting it.
distinct_grain = written.select("station_id", "start_date", "hour").distinct().count()
dupes = n - distinct_grain
print(f"duplicate station-hours : {dupes:,}")
checks.append(("one row per station-hour", dupes == 0))

# the target is never null — zero-filled hours are 0, not missing
checks.append(("trip_count not null", written.where(F.col("trip_count").isNull()).count() == 0))

# lags are null only at the head of each station's history, never later
late_null = written.where(
    (F.col("lag_168h").isNull())
    & (F.col("start_date") > F.date_add(F.lit(bounds.min_date), 7))
).count()
checks.append(("lag_168h null only at head", late_null == 0))

# Recompute lag_24h from the written table and compare. A grid fan-out leaves the
# window ordering ambiguous and silently halves the reach of every lag/rangeBetween
# — invisible in row counts, but it wrecks the features. This is the check that
# would have caught the 36% corrupted lags directly.
recheck = Window.partitionBy("station_id").orderBy("start_date", "hour")
lag_mismatch = (
    written.withColumn("expected_lag_24h", F.lag("trip_count", 24).over(recheck))
    .where(F.col("expected_lag_24h").isNotNull())
    .where(F.col("lag_24h") != F.col("expected_lag_24h"))
    .count()
)
print(f"lag_24h mismatches : {lag_mismatch:,}")
checks.append(("lag_24h matches recomputed", lag_mismatch == 0))

for name, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

assert all(passed for _, passed in checks), "featured/hourly failed validation"

print("\ntarget summary")
written.select("trip_count").describe().show()

print("null counts per feature")
written.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ML_COLUMNS]
).show(vertical=True, truncate=False)